# V2 Recomendação baseada em histórico: abordagem híbrida

Explorou-se o
*what-if* do enunciado: como mudaria a recomendação se tivéssemos o **histórico
de compras** de cada cliente (usando *sequential recommendation*), e como tratar
clientes **novos** (cold start).

## 1. As tabelas sintéticas

Gerou-se as tabelas `dim_customers` e `orders` a partir da matriz de co-ocorrência real. As proporções de venda por produto seguem o `times_sold` real. A atribuição a clientes e a sequência temporal (`order_seq`) são fabricadas.

In [5]:
from history_v2 import build_history, recommend_from_history

dim_customers, orders = build_history()
print(f"{len(dim_customers)} clientes | {orders['order_id'].nunique()} orders | {len(orders)} linhas")
dim_customers.head()


500 clientes | 3113 orders | 4471 linhas


,customer_id,customer_name,segment
0,1,Cliente_0001,entusiasta
1,2,Cliente_0002,entusiasta
2,3,Cliente_0003,casual
3,4,Cliente_0004,casual
4,5,Cliente_0005,casual


In [ ]:
orders.head(8)


,order_id,customer_id,product,order_seq
0,1,51,Super Mario Odyssey,1
1,2,130,Zelda: Breath of the Wild,1
2,3,264,Nintendo Switch,1
3,3,264,Nintendo Switch Dock Set,1
4,4,68,Sonic Mania,1
5,5,238,Sonic Generations,1
6,6,59,Animal Crossing: New Horizons,1
7,7,394,Animal Crossing: New Horizons,1


## 2. O que a v2 desbloqueia face à v1

A v1 trabalha com **co-ocorrência agregada** (simétrica, sem tempo). A v2 acrescenta
duas coisas que mudam os métodos possíveis:

- **Histórico individual**: sabemos o que *cada cliente* comprou.
- **Sequência temporal** (`order_seq`): sabemos a *ordem* das compras.

Isto abre a porta a métodos que a v1 não pode usar: desde user-based collaborative filtering, até **sequential recommendation** (a ordem importa).

## 3. A proposta: sistema híbrido com *routing* por estado do cliente

O desafio central com histórico é o **cold start**: um cliente novo não tem
histórico, logo um modelo baseado em histórico não tem o que usar. A solução madura
não é um modelo único — é **escolher o método consoante o estado do cliente**.

### 3.1 Os dois métodos

| Estado do cliente | Sinal disponível | Método adequado |
|---|---|---|
| **Novo** (0 / pouquíssimas compras) | atributos do produto, popularidade | **Content-based + popularidade** (cenário cold start) |
| **Com histórico** | sequência de compras | **Sequential / colaborativo** (a ordem e o padrão importam) |

## 4. Demonstração mínima dos dois métodos

Para tornar tangível (sobre os dados sintéticos): um cliente **com histórico**
recebe recomendações com base nas suas compras; um cliente **sem histórico**
cai no fallback de popularidade.

In [ ]:
# Cliente com histórico
ativo = orders["customer_id"].value_counts().index[0]
rec = recommend_from_history(ativo, dim_customers, orders)
print(f"Cliente {ativo} (com histórico):")
print("  Histórico:", rec["history"][:5], "...")
print("  Recomenda:", rec["recommendations"])


Cliente 282 (com histórico):
  Histórico: ['Animal Crossing: New Horizons', 'Splatoon 3', 'Nintendo Switch Pro Controller', 'Nintendo Switch', 'Pikmin 4'] ...
  Recomenda: ['Joy-Con Controllers (Pair)', 'Mario Party Superstars', 'Sonic Mania']


In [ ]:
# Cliente sem histórico (cold start)
novo = 999999
rec_novo = recommend_from_history(novo, dim_customers, orders)
print(f"Cliente {novo} (novo, sem histórico):")
print("  Recomenda:", rec_novo["recommendations"])
print("  Nota:", rec_novo["note"])

# Fallback de popularidade que um sistema híbrido usaria para o cold start:
populares = orders["product"].value_counts().head(3).index.tolist()
print("  → Fallback de popularidade (cold start):", populares)


Cliente 999999 (novo, sem histórico):
  Recomenda: []
  Nota: Cliente sem histórico (cold start) — usaria popularidade global.
  → Fallback de popularidade (cold start): ['Animal Crossing: New Horizons', 'Splatoon 3', 'Nintendo Switch']


## 5. Próximos passos (o que faria com dados reais)

Sobre dados reais de histórico (não sintéticos), a evolução seria:

1. **Baseline:** popularidade + co-ocorrência.
2. **Modelo principal:** *matrix factorization* (ALS/BPR) sobre cliente×produto.
3. **Sequencial:** e a ordem dos dados for preditiva: cadeia de Markov como baseline.
4. **Híbrido com transição ponderada** para tratar cold start.
5. **Avaliação temporal:** treino no passado, teste no futuro.